# Изучение развития игровой индустрии в 2000-2013 гг.

# Цели и задачи проекта

В этой тетрадке мы используем датасет `new_games.csv`, который содержит информацию о продажах игр разных жанров и платформ, к какой категории они относятся, а также пользовательские и экспертные оценки игр. 

Задача проекта - познакомиться с данными, проверить их корректность и провести предобработку, получить необходимый срез данных, распределить игры по категориям в зависимости от того, какую оценку они получили от пользователей и критиков. Определить платформы, на которых выпускалось самое большое количество игр.

# Описание данных

Данные `/datasets/new_games.csv` содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:

- `Name` — название игры.

- `Platform` — название платформы.

- `Year of Release` — год выпуска игры.

- `Genre` — жанр игры.

- `NA sales` — продажи в Северной Америке (в миллионах проданных копий).

- `EU sales` — продажи в Европе (в миллионах проданных копий).

- `JP sales` — продажи в Японии (в миллионах проданных копий).

- `Other sales` — продажи в других странах (в миллионах проданных копий).

- `Critic Score` — оценка критиков (от 0 до 100).

- `User Score` — оценка пользователей (от 0 до 10).

- `Rating` — рейтинг организации ESRB (англ. Entertainment Software Rating Board).

# Содержимое проекта

* [Загрузка данных и знакомство с данными](#1-bullet)
* [Проверка ошибок в данных и их предобработка](#2-bullet)
* [Фильтрация данных](#3-bullet)
* [Категоризация данных](#4-bullet)
* [Итоговый вывод](#5-bullet)

## Загрузка данных и знакомство с данными


In [1]:
# Импортируем библиотеку pandas
import pandas as pd

In [2]:
# Выгружаем данные из датасета new_games.csv в датафрейм games
games = pd.read_csv('/datasets/new_games.csv')

In [3]:
# Выводим информацию о датафрейме
games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [4]:
# Выводим первые строки датафрейма на экран
games.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


▎Промежуточный вывод

Датасет `new_games.csv` содержит 11 столбцов и 16956 строк, в которых представлена информация о продажах и оценках игр на разных платформах.

Изучим типы данных и их корректность:
- **Числовые значения с плавающей запятой (float64).** Четыре столбца имеют тип данных `float64`: `Year of Release`, `NA sales`, `Other sales`, `Critic Score`.
- **Строковые данные (object).** Семь столбцов имеют тип данных `object`: `Name`, `Platform`, `Genre`, `EU sales`, `JP sales`, `User Score`, `Rating`.   

Разберем столбцы по порядку:
- `Name` содержит строковую информацию с названием игры. Тип `object` выбран верно.
- `Platform` содержит строковую информацию с названием платформы, для которой выпущена игра. Тип `object` выбран верно.
- `Year of Release` содержит информацию о годе выпуска игры, в столбце есть пропуски строк. Тип `float64` выбран верно.
- `Genre` содержит строковую информацию о жанре игры. Тип `object` выбран верно.
- `NA sales` содержит информацию о продажах игр. Тип `float64` выбран верно.
- `EU sales`, `JP sales` содержат информацию о продажах игр. Тип `object` выбран неверно, следует изменить на `float64`.
- `Critic Score` содержит информацию об оценках игр среди критиков. Тип `float64` выбран верно.
- `User Score` содержит информацию об оценках игр среди пользователей. Тип `object` выбран неверно, следует изменить на `float64`.
- `Rating` содержит информацию о том, к какому рейтингу относится игра. Это категориальный признак, но для дальнейшей работы достаточно типа `object` — переводить в `category` не будем, чтобы не усложнять обработку пропусков и нормализацию.

Во всех столбцах, для которых выбран числовой тип с плавающей запятой, можно понизить битовый разряд до `float32`, что положительно скажется на объеме занимаемой памяти файла.

После анализа типов данных видно, что 4 из 11 столбцов представлены некорректно. Для оптимизации можно использовать числовые значения с плавающей запятой и категориальный тип данных.

Названия столбцов соответствуют их описанию, но для  дальнейшего использования стоит их перевести в нижний регистр и привести к стилю snake case.

##  Проверка ошибок в данных и их предобработка


### Названия, или метки, столбцов датафрейма

In [5]:
# Отображение названий всех столбцов
print("Названия столбцов:")
games.columns

Названия столбцов:


Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

C целью улучшения читаемости кода и устранения потенциальных ошибок приведем все названия к стилю `snake_case`

In [6]:
# Приведение названий столбцов к стилю snake_case
games.columns = games.columns.str.lower().str.replace(' ', '_')

In [7]:
# Выведем на экран новые названия
print("Изменённые названия столбцов:")
games.columns

Изменённые названия столбцов:


Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

In [8]:
games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16956 non-null  object 
 6   jp_sales         16956 non-null  object 
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       10152 non-null  object 
 10  rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


### Типы данных

In [9]:
# Создаю список из столбцов, которые нужно будет перевести к float32
columns_to_convert = [
    'year_of_release',
    'na_sales',
    'eu_sales',
    'jp_sales',
    'user_score',
    'other_sales',
    'critic_score'
]
# Приведение столбцов к float с понижением битности до 32
games[columns_to_convert] = games[columns_to_convert].apply(lambda x: pd.to_numeric(x, errors='coerce', downcast='float'))

In [10]:
games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float32
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float32
 5   eu_sales         16950 non-null  float32
 6   jp_sales         16952 non-null  float32
 7   other_sales      16956 non-null  float32
 8   critic_score     8242 non-null   float32
 9   user_score       7688 non-null   float32
 10  rating           10085 non-null  object 
dtypes: float32(7), object(4)
memory usage: 993.6+ KB


- Уменьшение битности привело к снижению объема занимаемого файла с 1,4 Мб до 878,1 Кб.

*В контексте выполняемой задачи (выборка с 2000 по 2013 года) можно было преобразовать `year_of_release` к `int8` (удалив данные, которые вызовут ошибку при выполнении кода), но на случай, если эти данные еще могут пригодиться в будущем, преобразовывать их не стал.*

### Наличие пропусков в данных


In [11]:
# Подсчитываем абсолютное количество строк с пропусками
games.isna().sum()

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

In [12]:
# Подсчитываем относительное количество (процент) строк с пропусками
games.isna().sum() / len(games) * 100

name                0.011795
platform            0.000000
year_of_release     1.621845
genre               0.011795
na_sales            0.000000
eu_sales            0.035386
jp_sales            0.023590
other_sales         0.000000
critic_score       51.391838
user_score         54.659118
rating             40.522529
dtype: float64

In [13]:
# Выведем таблицу со значением доли пропусков в абсолютном и относительном значении
missing_table = pd.DataFrame({
    'Количество пропусков': games.isnull().sum(),
    'Доля пропусков (%)': (games.isnull().mean() * 100).round(2)
})

missing_table = missing_table.sort_values('Количество пропусков', ascending=False)
missing_table

,Количество пропусков,Доля пропусков (%)
user_score,9268,54.66
critic_score,8714,51.39
rating,6871,40.52
year_of_release,275,1.62
eu_sales,6,0.04
jp_sales,4,0.02
name,2,0.01
genre,2,0.01
platform,0,0.00
na_sales,0,0.00


▎Промежуточный вывод

В представленных данных есть пропуски в следующих столбцах:

1. `name`: 2 пропуска
2. `year_of_release`: 275 пропусков
3. `genre`: 2 пропуска
4. `eu_sales`: 6 пропусков
5. `jp_sales`: 4 пропуска
6. `critic_score`: 8714 пропусков
7. `user_score`: 9268 пропусков
8. `rating`: 6871 пропусков

▎Причины пропусков:

• `name` и `genre`: Возможно, некоторые игры не имели указанных значений по причине ошибок при сборе данных или отсутствия информации о названии/жанре.

• `year_of_release`: Пропуски могут быть связаны с тем, что для некоторых игр не была указана дата выхода, что может произойти, если игра была анонсирована, но еще не выпущена.

• `eu_sales`, `jp_sales`: Пропуски могут возникнуть из-за отсутствия данных о продажах в определенных регионах, возможно, из-за того, что игра не была выпущена в этих регионах или данные не были собраны.

• `critic_score` и `user_score`: Высокое количество пропусков может указывать на то, что для многих игр не было рецензий или пользователей, оставивших оценки.

Действия с пропущенными данными

1. `name` и `genre`: Поскольку это важные идентификаторы для анализа, эти строки лучше удалить, чтобы избежать путаницы.

2. `year_of_release`: Пропуски оставим как есть — игры без года выпуска не войдут в срез 2000–2013 гг. на шаге фильтрации, и это логично.

3. `eu_sales`, `jp_sales`: Пропусков немного (6 и 4 соответственно). Заполним их средними значениями в зависимости от платформы и года выхода игры — это не исказит результаты.

4. `critic_score` и `user_score`: Можно было бы заменить пропуски на средние значения по всем играм или по жанру/платформе, но с учетом поставленной задачи это исказит исследование. Оставляем без изменений. 

5. `rating`: Заполним пропуски значением "Not Rated" — это корректный маркер отсутствия рейтинга.

In [14]:
# Удаление строк с пропусками в 'name' и 'genre'
games.dropna(subset=['name', 'genre'], inplace=True)

In [15]:
# Замена пропусков в 'eu_sales' и 'jp_sales' на средние значения по платформе и году выхода
mean_sales = games.groupby(['platform', 'year_of_release'])[['eu_sales', 'jp_sales']].mean().reset_index()

In [16]:
# Объединяем данные для замены
games = games.merge(mean_sales, on=['platform', 'year_of_release'], suffixes=('', '_mean'), how='left')

In [17]:
# Заполняем пропуски
games['eu_sales'].fillna(games['eu_sales_mean'], inplace=True)
games['jp_sales'].fillna(games['jp_sales_mean'], inplace=True)

In [18]:
# Удаляем вспомогательные столбцы
games.drop(columns=['eu_sales_mean', 'jp_sales_mean'], inplace=True)

In [19]:
# Заполняем пропуски в 'rating'
games['rating'].fillna('Not Rated', inplace=True)

In [20]:
# Проверка на наличие оставшихся пропусков
print(games.isnull().sum())

name                  0
platform              0
year_of_release     275
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8712
user_score         9266
rating                0
dtype: int64


Посмотрим, как сейчас выглядит датасет

In [21]:
games.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16954 entries, 0 to 16953
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16954 non-null  object 
 2   year_of_release  16679 non-null  float32
 3   genre            16954 non-null  object 
 4   na_sales         16954 non-null  float32
 5   eu_sales         16954 non-null  float32
 6   jp_sales         16954 non-null  float32
 7   other_sales      16954 non-null  float32
 8   critic_score     8242 non-null   float32
 9   user_score       7688 non-null   float32
 10  rating           16954 non-null  object 
dtypes: float32(7), object(4)
memory usage: 1.1+ MB


- Размерность датасета увеличилась ввиду заполнения строк для части столбцов

### Явные и неявные дубликаты в данных

In [22]:
# Изучаем уникальные значения в колонках 'genre', 'platform', 'rating' и 'year_of_release'
print("Уникальные значения в жанрах:")
games['genre'].unique()

Уникальные значения в жанрах:


array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', 'MISC', 'ROLE-PLAYING', 'RACING', 'ACTION', 'SHOOTER',
       'FIGHTING', 'SPORTS', 'PLATFORM', 'ADVENTURE', 'SIMULATION',
       'PUZZLE', 'STRATEGY'], dtype=object)

In [23]:
print("Уникальные значения в платформах:")
games['platform'].unique()

Уникальные значения в платформах:


array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne',
       'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

In [24]:
print("Уникальные значения в рейтингах:")
games['rating'].unique()

Уникальные значения в рейтингах:


array(['E', 'Not Rated', 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP'],
      dtype=object)

Помимо новой категории `'Not Rated'` в списке представлена категория `'K-А'`, которая не представлена в задании
- `K-A (Kids to Adults)` — рейтинг «Для детей и взрослых» в системе Entertainment Software Rating Board (ESRB)

In [25]:
print("Уникальные значения в годах выпуска:")
games['year_of_release'].unique()

Уникальные значения в годах выпуска:


array([2006., 1985., 2008., 2009., 1996., 1989., 1984., 2005., 1999.,
       2007., 2010., 2013., 2004., 1990., 1988., 2002., 2001., 2011.,
       1998., 2015., 2012., 2014., 1992., 1997., 1993., 1994., 1982.,
       2016., 2003., 1986., 2000.,   nan, 1995., 1991., 1981., 1987.,
       1980., 1983.], dtype=float32)

In [26]:
# Проверяем наличие неявных дубликатов в жанрах, платформах и рейтингах
print("Количество уникальных значений в жанрах:", games['genre'].nunique())
print("Количество уникальных значений в платформах:", games['platform'].nunique())
print("Количество уникальных значений в рейтингах:", games['rating'].nunique())

Количество уникальных значений в жанрах: 24
Количество уникальных значений в платформах: 31
Количество уникальных значений в рейтингах: 9


В столбце жанров (`genre`) есть дубликаты названий жанров в связи с тем, что часть значений представлена в ВЕРХНЕМ РЕГИСТРЕ. 

Приведем столбцы названия и жанра игры к нижнему регистру, а столбец рейтинга — к верхнему.

In [27]:
# Нормализация данных
# Приведём названия и жанры к нижнему регистру
games['name'] = games['name'].str.lower()
games['genre'] = games['genre'].str.lower()

In [28]:
# Приведём названия рейтингов к верхнему регистру
games['rating'] = games['rating'].str.upper()

In [29]:
# Проверяем уникальные значения после нормализации
print("Уникальные значения в жанрах после нормализации:")
print("Количество уникальных значений в жанрах:", games['genre'].nunique())
games['genre'].unique()

Уникальные значения в жанрах после нормализации:
Количество уникальных значений в жанрах: 12


array(['sports', 'platform', 'racing', 'role-playing', 'puzzle', 'misc',
       'shooter', 'simulation', 'action', 'fighting', 'adventure',
       'strategy'], dtype=object)

In [30]:
print("Уникальные значения в платформах после нормализации:")
print("Количество уникальных значений в платформах:", games['platform'].nunique())
games['platform'].unique()

Уникальные значения в платформах после нормализации:
Количество уникальных значений в платформах: 31


array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne',
       'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

In [31]:
print("Уникальные значения в рейтингах после нормализации:")
print("Количество уникальных значений в рейтингах:", games['rating'].nunique())
games['rating'].unique()

Уникальные значения в рейтингах после нормализации:
Количество уникальных значений в рейтингах: 9


array(['E', 'NOT RATED', 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP'],
      dtype=object)

In [32]:
# Замена значения "K-A" на "E" в столбце 'rating'
games['rating'] = games['rating'].replace('K-A', 'E')

# Проверка уникальных значений после замены
print("Уникальные значения в рейтингах после замены 'K-A' на 'E':")
print("Количество уникальных значений в рейтингах:", games['rating'].nunique())
games['rating'].unique()

Уникальные значения в рейтингах после замены 'K-A' на 'E':
Количество уникальных значений в рейтингах: 8


array(['E', 'NOT RATED', 'M', 'T', 'E10+', 'AO', 'EC', 'RP'], dtype=object)

In [33]:
# Проверяем наличие явных дубликатов в данных
duplicate_rows = games[games.duplicated()]

In [34]:
# Выводим количество явных дубликатов
print("Количество явных дубликатов:")
duplicate_rows.shape[0]

Количество явных дубликатов:


241

In [35]:
# Если есть явные дубликаты, выводим их
if not duplicate_rows.empty:
    print("Явные дубликаты:")
duplicate_rows

Явные дубликаты:


,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
268,batman: arkham asylum,PS3,2009.0,action,2.24,1.31,0.07,0.61,91.0,8.9,T
368,james bond 007: agent under fire,PS2,2001.0,shooter,1.90,1.13,0.10,0.41,72.0,7.9,T
716,god of war: ascension,PS3,2013.0,action,1.23,0.63,0.04,0.35,80.0,7.5,M
822,wipeout: the game,Wii,2009.0,misc,1.94,0.00,0.00,0.12,NaN,NaN,NOT RATED
847,rayman raving rabbids: tv party,Wii,2008.0,misc,0.72,1.08,0.00,0.23,73.0,7.7,E10+
...,...,...,...,...,...,...,...,...,...,...,...
16669,fullmetal alchemist: prince of the dawn,Wii,2009.0,adventure,0.00,0.00,0.01,0.00,NaN,NaN,NOT RATED
16751,routes pe,PS2,2007.0,adventure,0.00,0.00,0.01,0.00,NaN,NaN,NOT RATED
16797,transformers: prime,Wii,2012.0,action,0.00,0.01,0.00,0.00,NaN,NaN,NOT RATED
16910,metal gear solid v: the definitive experience,XOne,2016.0,action,0.01,0.00,0.00,0.00,NaN,NaN,M


In [36]:
# Удаляем явные дубликаты и сохраняем результат в основной датафрейм
games = games.drop_duplicates()

In [37]:
# Проверяем количество строк после удаления дубликатов
games.shape[0]

16713

In [38]:
games.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16713 entries, 0 to 16953
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16713 non-null  object 
 1   platform         16713 non-null  object 
 2   year_of_release  16444 non-null  float32
 3   genre            16713 non-null  object 
 4   na_sales         16713 non-null  float32
 5   eu_sales         16713 non-null  float32
 6   jp_sales         16713 non-null  float32
 7   other_sales      16713 non-null  float32
 8   critic_score     8137 non-null   float32
 9   user_score       7590 non-null   float32
 10  rating           16713 non-null  object 
dtypes: float32(7), object(4)
memory usage: 1.1+ MB


▎Промежуточный вывод

В процессе анализа данных было выявлено наличие явных дубликатов. 

• Количество найденных явных дубликатов: 241

Для обработки этих дубликатов были предприняты следующие действия:

1. `Идентификация дубликатов`: Все строки, которые полностью совпадали с другими строками в датафрейме, были определены с помощью метода duplicated().

2. `Анализ дубликатов`: Выявленные дубликаты были выведены на экран для дальнейшего анализа, чтобы убедиться в их идентичности и понять причины их появления.

3. `Удаление дубликатов`: Все явные дубликаты были удалены из датафрейма с помощью метода drop_duplicates(), что позволило очистить данные и избежать искажения результатов анализа.

4. `Обновление данных`: После удаления дубликатов общее количество строк в датафрейме уменьшилось до 16713.

Таким образом, данные были успешно очищены от явных дубликатов, что способствует более точному анализу и интерпретации информации.

In [39]:
# Исходное число строк до удаления дубликатов
initial_rows = 16956
initial_rows

16956

In [40]:
# Количество строк после удаления явных дубликатов
cleaned_rows = 16713
cleaned_rows

16713

In [41]:
# Абсолютное число удалённых строк
deleted_rows_abs = initial_rows - cleaned_rows
deleted_rows_abs

243

In [42]:
# Относительное число удалённых строк в %
deleted_rows_rel = deleted_rows_abs / initial_rows * 100
round(deleted_rows_rel, 2)

1.43

In [43]:
games.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16713 entries, 0 to 16953
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16713 non-null  object 
 1   platform         16713 non-null  object 
 2   year_of_release  16444 non-null  float32
 3   genre            16713 non-null  object 
 4   na_sales         16713 non-null  float32
 5   eu_sales         16713 non-null  float32
 6   jp_sales         16713 non-null  float32
 7   other_sales      16713 non-null  float32
 8   critic_score     8137 non-null   float32
 9   user_score       7590 non-null   float32
 10  rating           16713 non-null  object 
dtypes: float32(7), object(4)
memory usage: 1.1+ MB


▎Промежуточный вывод

По результатам проделанной работы было выявлено 241 явный дубликат, что составляет 1,42% от общего количества строк.

## Фильтрация данных

In [44]:
# Фильтруем данные по году выпуска
df_actual = games[(games['year_of_release'] >= 2000) & (games['year_of_release'] <= 2013)]

In [45]:
# Выводим информацию о новом датафрейме
df_actual.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12781 entries, 0 to 16952
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             12781 non-null  object 
 1   platform         12781 non-null  object 
 2   year_of_release  12781 non-null  float32
 3   genre            12781 non-null  object 
 4   na_sales         12781 non-null  float32
 5   eu_sales         12781 non-null  float32
 6   jp_sales         12781 non-null  float32
 7   other_sales      12781 non-null  float32
 8   critic_score     7169 non-null   float32
 9   user_score       6483 non-null   float32
 10  rating           12781 non-null  object 
dtypes: float32(7), object(4)
memory usage: 848.7+ KB


▎Промежуточный вывод

- В общий расчет исследования вошла 12781 строка.

---

## Категоризация данных

In [46]:
df_actual_1 = df_actual.copy()

In [47]:
# Отобразим значения оценок пользователей
df_actual_1['user_score'].unique()

array([8. , 8.3, 8.5, 6.6, 8.4, nan, 8.6, 7.7, 6.3, 7.4, 8.2, 9. , 7.9,
       8.1, 8.7, 7.1, 3.4, 5.3, 4.8, 3.2, 8.9, 6.4, 7.8, 7.5, 2.6, 7.2,
       7. , 7.3, 7.6, 9.1, 8.8, 4.3, 6.9, 6.8, 6.7, 6.1, 4. , 6.5, 5.4,
       9.3, 4.2, 6. , 3.7, 6.2, 5.8, 5.6, 9.2, 4.6, 5.9, 5.7, 5.5, 4.9,
       2.9, 9.4, 5.2, 3.3, 4.1, 5. , 4.4, 4.5, 3.5, 2.5, 1.9, 3.1, 2.2,
       2. , 9.5, 5.1, 4.7, 2.1, 3.6, 2.8, 1.8, 3.8, 3. , 0. , 1.6, 9.6,
       3.9, 1.7, 2.7, 1.1, 0.3, 2.4, 1.2, 2.3, 0.5, 1.3, 0.6, 1.4, 0.9,
       1. , 1.5, 0.7, 0.2, 9.7], dtype=float32)

In [48]:
# Напишем функцию для категоризации игр по оценкам пользователей
def categorize_user_score(score):
    if 8 <= score <= 10:
        return 'высокая оценка'
    elif 3 <= score < 8:
        return 'средняя оценка'
    elif 0 <= score < 3:
        return 'низкая оценка'
    else:
        return 'не определено'

In [49]:
df_actual_1['user_score_category'] = df_actual_1['user_score'].apply(categorize_user_score)

In [50]:
# Отобразим значения оценок критиков
print(df_actual_1['critic_score'].unique())

[76. 82. 80. 89. 58. 87. nan 91. 61. 97. 95. 77. 88. 83. 94. 93. 85. 86.
 98. 90. 84. 73. 74. 78. 71. 72. 96. 68. 62. 92. 49. 67. 81. 66. 56. 79.
 59. 64. 70. 60. 63. 75. 69. 50. 25. 42. 44. 55. 48. 57. 29. 47. 65. 54.
 20. 53. 37. 38. 33. 52. 30. 32. 43. 45. 51. 40. 46. 39. 34. 41. 36. 31.
 27. 35. 26. 19. 28. 23. 24. 21. 17. 13.]


In [51]:
# Напишем функцию для категоризации игр по оценкам критиков
def categorize_critic_score(score):
    if 80 <= score <= 100:
        return 'высокая оценка'
    elif 30 <= score < 80:
        return 'средняя оценка'
    elif 0 <= score < 30:
        return 'низкая оценка'
    else:
        return 'не определено'

In [52]:
df_actual_1['critic_score_category'] = df_actual_1['critic_score'].apply(categorize_critic_score)

In [53]:
print("Распределение игр по категориям оценок пользователей:")
df_actual_1.groupby('user_score_category')['name'].count()

Распределение игр по категориям оценок пользователей:


user_score_category
высокая оценка    2286
не определено     6298
низкая оценка      116
средняя оценка    4081
Name: name, dtype: int64

In [54]:
print("Распределение игр по категориям оценок критиков:")
df_actual_1.groupby('critic_score_category')['name'].count()

Распределение игр по категориям оценок критиков:


critic_score_category
высокая оценка    1692
не определено     5612
низкая оценка       55
средняя оценка    5422
Name: name, dtype: int64

"Голые" цифры мало дают понимания результата анализа. Посчитаем относительно количество оценок пользователей от общего числа, исключив категорию `не определено`.

In [55]:
# Исключаем строки с категорией 'не определено'
user_df_filtered = df_actual_1[df_actual_1['user_score_category'] != 'не определено']

# Группируем и считаем
user_category_counts = user_df_filtered['user_score_category'].value_counts()

# Рассчитываем относительные доли
user_total = user_category_counts.sum()
user_relative_counts = user_category_counts / user_total *100

print("Относительное количество оценок по категориям (без учета 'не определено'):")
round(user_relative_counts, 2)

Относительное количество оценок по категориям (без учета 'не определено'):


средняя оценка    62.95
высокая оценка    35.26
низкая оценка      1.79
Name: user_score_category, dtype: float64

Аналогично для оценок критиков:

In [56]:
# Исключаем строки с категорией 'не определено'
critic_df_filtered = df_actual_1[df_actual_1['critic_score_category'] != 'не определено']

# Группируем и считаем
critic_category_counts = critic_df_filtered['critic_score_category'].value_counts()

# Рассчитываем относительные доли
critic_total = critic_category_counts.sum()
critic_relative_counts = critic_category_counts / critic_total *100

print("Относительное количество оценок по категориям (без учета 'не определено'):")
round(critic_relative_counts, 2)

Относительное количество оценок по категориям (без учета 'не определено'):


средняя оценка    75.63
высокая оценка    23.60
низкая оценка      0.77
Name: critic_score_category, dtype: float64

Для наглядности выведем первые 20 строк данных, в которых будут отображены значение оценок и категория оценки, к которой определена игра

In [57]:
df_actual_1[['user_score', 'user_score_category', 'critic_score', 'critic_score_category']].head(20)

,user_score,user_score_category,critic_score,critic_score_category
0,8.0,высокая оценка,76.0,средняя оценка
2,8.3,высокая оценка,82.0,высокая оценка
3,8.0,высокая оценка,80.0,высокая оценка
6,8.5,высокая оценка,89.0,высокая оценка
7,6.6,средняя оценка,58.0,средняя оценка
8,8.4,высокая оценка,87.0,высокая оценка
10,NaN,не определено,NaN,не определено
11,8.6,высокая оценка,91.0,высокая оценка
13,7.7,средняя оценка,80.0,высокая оценка
14,6.3,средняя оценка,61.0,средняя оценка


- После проведения категоризации, полученный столбцы можно преобразовать в формат `'category'`

In [58]:
# Оптимизация типов:
df_actual_1['user_score_category'] = df_actual_1['user_score_category'].astype('category')
df_actual_1['critic_score_category'] = df_actual_1['critic_score_category'].astype('category')

In [59]:
df_actual_1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12781 entries, 0 to 16952
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   name                   12781 non-null  object  
 1   platform               12781 non-null  object  
 2   year_of_release        12781 non-null  float32 
 3   genre                  12781 non-null  object  
 4   na_sales               12781 non-null  float32 
 5   eu_sales               12781 non-null  float32 
 6   jp_sales               12781 non-null  float32 
 7   other_sales            12781 non-null  float32 
 8   critic_score           7169 non-null   float32 
 9   user_score             6483 non-null   float32 
 10  rating                 12781 non-null  object  
 11  user_score_category    12781 non-null  category
 12  critic_score_category  12781 non-null  category
dtypes: category(2), float32(7), object(4)
memory usage: 874.1+ KB


In [60]:
# Группируем по платформам и считаем количество игр
platform_counts = df_actual_1.groupby('platform')['name'].count().sort_values(ascending=False)

# Выводим топ-7 платформ
platform_counts.head(7)

platform
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: name, dtype: int64

---

## Итоговый вывод


▎Подготовительная часть

В рамках проделанной работы был рассмотрен датасет `new_games.csv` из 11 столбцов, в который вошли 16956 строк. В нём была представлена информация о продажах и оценках игр на разных платформах.

В 3 из 11 столбцов было выявлено несоответствие типа данных (`eu_sales`, `jp_sales`, `user_score`), что было исправлено, а затем для числовых типов данных была уменьшена битность, что, в свою очередь, позволило уменьшить количество занимаемой памяти.

Названия столбцов соответствуют их описанию, но для дальнейшего использования они были переведены в нижний регистр в стиле `snake_case`.

По итогам исследования были выявлены пропуски в следующих столбцах: `name` — 2 пропуска, `year_of_release` — 275 пропусков, `genre` — 2 пропуска, `eu_sales` — 6 пропусков, `jp_sales` — 4 пропуска, `critic_score` — 8714 пропусков, `user_score` — 9268 пропусков, `rating` — 6871 пропуск. Для дальнейшей работы в зависимости от имеющихся данных строки с пропущенными значениями были удалены (`name` и `genre`), пропуски в `eu_sales` и `jp_sales` заполнены средними значениями по платформе и году выхода, пропуски в `critic_score` и `user_score` оставлены без изменений, а для `year_of_release` пропуски сохранены — такие игры не войдут в срез 2000–2013 гг. Для столбца `rating` пропуски заполнены значением `Not Rated`.

Было выявлено наличие явных дубликатов (241), которые были удалены, что составило 1,42% от общего количества строк. После удаления дубликатов общее количество строк в датасете уменьшилось до 16713.

▎Исследовательская часть

Для изучения истории продаж в рамках заданных условий были отобраны игры, которые выпускались с 2000 по 2013 гг. включительно. В новый датасет с учётом удалённых явных дубликатов вошла 12781 игра.

Далее была проведена категоризация по оценкам пользователей `user_score` и критиков `critic_score`. В ходе предобработки пропуски в этих столбцах не были заполнены с целью избежания искажения информации. Для 6298 из них не были определены оценки среди пользователей, а для 5612 — среди критиков, что составляет примерно половину от наших данных.

За вычетом отсутствующих данных можно сделать вывод, что пользователи чаще отмечают игры высокой оценкой (в 35% случаев против 24% у критиков), но и чаще ставят плохие оценки (1,79% против 0,77% соответственно). Большинство оценок попали в среднюю категорию, что логично, исходя из условий категоризации.

Что же касается платформ, на которые выпускались игры в XXI веке, то здесь 1 и 2 место со значительным отрывом от остальных заняли `PS2` (PlayStation 2) и `DS` (Nintendo DS).

- В случае с PlayStation стоит учитывать, что в топ-7 попали также `PS3` и портативная `PSP`. Другими словами, более 34% игр в указанный период выпускалось для этой компании.

- У Nintendo в рейтинге помимо `DS` представлена приставка `GBA`. Для них было выпущено почти 23% игр из представленного датасета.

- На остальные компании приходится 43% игр.